In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import torch.nn.functional as F
import re
import random

In [22]:
_model_cache = {}

In [23]:
class SentimentAnalyzer:
    def __init__(self, model_name="cardiffnlp/twitter-roberta-base-sentiment"):
        global _model_cache
        if model_name not in _model_cache:
            _model_cache[model_name] = {
                "tokenizer": AutoTokenizer.from_pretrained(model_name, from_tf=True),
                "model": AutoModelForSequenceClassification.from_pretrained(model_name),
            }
        self.tokenizer = _model_cache[model_name]["tokenizer"]
        self.model = _model_cache[model_name]["model"]
        self.labels = ['negative', 'neutral', 'positive']

    def analyze(self, text):
        encoded_input = self.tokenizer(text, return_tensors='pt')
        with torch.no_grad():
            output = self.model(**encoded_input)
        scores = output.logits[0].numpy()
        probs = F.softmax(torch.tensor(scores), dim=0)
        top_class = torch.argmax(probs).item()
        return {
            'label': self.labels[top_class],
            'confidence': round(probs[top_class].item(), 3),
            'all_scores': dict(zip(self.labels, [round(p.item(), 3) for p in probs]))
        }


In [24]:
class EmotionClassifier:
    def __init__(self, model_name="bhadresh-savani/distilbert-base-uncased-emotion"):
        global _model_cache
        if model_name not in _model_cache:
            _model_cache[model_name] = {
                "tokenizer": AutoTokenizer.from_pretrained(model_name),
                "model": AutoModelForSequenceClassification.from_pretrained(model_name),
            }
        self.tokenizer = _model_cache[model_name]["tokenizer"]
        self.model = _model_cache[model_name]["model"]
        self.labels = [
            'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
            'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
            'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
            'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
            'relief', 'remorse', 'sadness', 'surprise', 'neutral'
        ]

    def classify(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
            scores = F.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(scores).item()
        confidence = torch.max(scores).item()
        return {
            "label": self.labels[predicted_class],
            "confidence": round(confidence * 100, 2)
        }


In [25]:
class BiasDetector:
    def __init__(self, model_name="unitary/toxic-bert", threshold=0.7):
        global _model_cache
        if model_name not in _model_cache:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
            _model_cache[model_name] = {
                "tokenizer": tokenizer,
                "model": model,
                "pipeline": pipe
            }
        self.tokenizer = _model_cache[model_name]["tokenizer"]
        self.model = _model_cache[model_name]["model"]
        self.pipeline = _model_cache[model_name]["pipeline"]
        self.threshold = threshold

    def detect(self, text):
        result = self.pipeline(text)[0]
        if result['score'] >= self.threshold and result['label'] == 'TOXIC':
            return True, "Biased or Manipulative (TOXIC)"
        return False, "Likely Neutral"


In [26]:
class NigerianPatterns:
    NIGERIAN_TRIGGER_PHRASES = [
        r"breaking[:\-]?", r"shocking truth", r"aswear", r"they don’t want you to know",
        r"na them", r"this country is finished", r"government is lying", r"our leaders have failed us",
        r"dem wan kill us", r"nawa o", r"wetin be this", r"shey na joke", r"this is madness"
    ]

    @staticmethod
    def has_misleading_pattern(text):
        text = text.lower()
        for pattern in NigerianPatterns.NIGERIAN_TRIGGER_PHRASES:
            if re.search(pattern, text):
                return True
        return False


In [27]:
class FakeNewsDetector:
    FAKE_PATTERNS = [
        r"\bBreaking\b", r"\bShocking\b", r"\bRevealed\b", r"\bExposed\b",
        r"\bScandal\b", r"what they (don’t|don't) want you to know",
        r"\bCollapse\b", r"\bTotal failure\b", r"Wake up Nigeria", r"\bMust watch\b",
        r"\bAgenda\b", r"\bHidden truth\b"
    ]

    @staticmethod
    def detect(text):
        matched_phrases = []
        for pattern in FakeNewsDetector.FAKE_PATTERNS:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                matched_phrases.extend(matches)
        return len(matched_phrases) > 0, matched_phrases


In [28]:
class TrustScoreCalculator:
    DID_YOU_KNOW_TIPS = [
        "Fake news often uses words like 'BREAKING' or 'SHOCKING' to create urgency.",
        "Subtle bias can be hidden in adjectives and tone, not just facts.",
        "Emotional posts get more shares—even if they’re false.",
        "Sensational headlines are more likely to spread misinformation.",
        "Repetition of a lie can make people believe it—even without proof.",
        "Manipulative language often appeals to fear, anger, or patriotism."
    ]

    @staticmethod
    def get_trust_indicator(score):
        if score >= 70:
            return "🟢 Trusted"
        elif 40 <= score < 70:
            return "🟡 Caution"
        else:
            return "🔴 Risky"

    @staticmethod
    def calculate(bias_score, emotion_score, sentiment_label, text):
        nigerian_flag = NigerianPatterns.has_misleading_pattern(text)
        fake_flag, fake_phrases = FakeNewsDetector.detect(text)

        score = 100
        explanation = []

        # Bias penalty
        if bias_score >= 0.8:
            score -= 30
            explanation.append("Text shows strong biased or one-sided language.")
        elif bias_score >= 0.6:
            score -= 20
            explanation.append("Text shows moderate bias.")
        elif bias_score >= 0.4:
            score -= 10
            explanation.append("Text shows mild bias.")

        # Emotion penalty
        if emotion_score >= 0.8:
            score -= 25
            explanation.append("Text is extremely emotionally charged.")
        elif emotion_score >= 0.6:
            score -= 15
            explanation.append("Text has strong emotional tone.")
        elif emotion_score >= 0.4:
            score -= 8
            explanation.append("Text has mild emotional tone.")

        # Sentiment penalty
        if sentiment_label == 'negative':
            score -= 10
            explanation.append("Text expresses a negative tone.")
        elif sentiment_label == 'positive':
            score -= 3
            explanation.append("Text expresses a positive tone.")
        else:
            explanation.append("Text appears to have a neutral sentiment.")

        # Nigerian manipulation pattern penalty
        if nigerian_flag:
            score -= 15
            explanation.append("Contains Nigerian expressions that are commonly used manipulatively.")

        # Fake news phrases penalty
        if fake_flag:
            score -= 10
            explanation.append(f"Text includes misleading phrases like: {', '.join(set(fake_phrases))}")

        # Clamp score between 0 and 100
        score = max(0, min(score, 100))
        indicator = TrustScoreCalculator.get_trust_indicator(score)
        did_you_know = random.choice(TrustScoreCalculator.DID_YOU_KNOW_TIPS)

        return score, indicator, explanation, did_you_know


In [34]:
def analyze(text):
    sentiment_analyzer = SentimentAnalyzer()
    emotion_classifier = EmotionClassifier()
    bias_detector = BiasDetector()
    
    sentiment = sentiment_analyzer.analyze(text)
    emotion = emotion_classifier.classify(text)
    bias_flag, bias_label = bias_detector.detect(text)
    
    score, indicator, explanation, tip = TrustScoreCalculator.calculate(
        bias_score=sentiment['all_scores']['negative'],  # or adjust to your bias measure
        emotion_score=emotion['confidence'] / 100,
        sentiment_label=sentiment['label'],
        text=text
    )
    
    print(f"Sentiment: {sentiment['label']}")
    print(f"Emotion: {emotion['label']}")
    print(f"Bias: {bias_label} (Flag: {bias_flag})")
    print(f"Trust Score: {score} ({indicator})")
    print("Explanation:")
    for line in explanation:
        print(f"- {line}")
    print(f"Did You Know? {tip}")


In [39]:
text = input('Headline or Text:')

analyze(text)


KeyboardInterrupt

